### LIBERO Dataset Comparison

This notebook compares two LIBERO spatial datasets:
- **Dataset 1 (ds)**: The original LIBERO spatial dataset in lerobot format
- **Dataset 2 (ds_collected)**: A newly collected LIBERO spatial dataset, that is collected by letting the Evo-1 robot performs actions (see Evo-1/LIBERO_evaluation/libero_client_4tasks.py)

The comparison includes:
1. **Structure comparison**: Features, data types, and number of samples
2. **Statistical comparison**: Min, max, mean, std for all features (observation.state, action, timestamp, frame_index, episode_index, index, task_index)


In [16]:
import numpy as np
import pandas as pd
from datasets import load_dataset

# Function to get statistics for a feature
def get_feature_stats(dataset, feature_name):
    """Extract statistics for a feature from the dataset"""
    data = dataset[feature_name]
    
    # Check if it's a list/array feature
    if isinstance(data[0], (list, np.ndarray)):
        # Flatten all arrays
        flattened = np.concatenate([np.array(x) for x in data])
        return {
            'type': 'array',
            'shape': f"{len(data[0])} elements per sample",
            'min': float(np.min(flattened)),
            'max': float(np.max(flattened)),
            'mean': float(np.mean(flattened)),
            'std': float(np.std(flattened)),
            'num_samples': len(data)
        }
    else:
        # Scalar feature
        arr = np.array(data)
        stats = {
            'type': 'scalar',
            'min': float(np.min(arr)),
            'max': float(np.max(arr)),
            'mean': float(np.mean(arr)),
            'std': float(np.std(arr)),
            'num_samples': len(arr)
        }
        
        # Add unique values if there are few
        unique_vals = np.unique(arr)
        if len(unique_vals) <= 20:
            stats['unique_values'] = unique_vals.tolist()
        else:
            stats['num_unique_values'] = len(unique_vals)
        
        return stats

In [ ]:
# Loading the 2 datasets
ds = load_dataset(
    "parquet",
    data_files="../libero/datasets/lerobot_format/libero_spatial_lerobot/data/*/*.parquet"
    )

ds_collected = load_dataset(
    "parquet",
    data_files="../../to_kill/libero_spatial/data/*/*.parquet"
)

DatasetDict({
    train: Dataset({
        features: ['observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'],
        num_rows: 52970
    })
})


In [17]:
# Compare structure
print("=" * 80)
print("STRUCTURE COMPARISON")
print("=" * 80)

print(f"\nDataset 1 (ds):")
print(f"  Features: {ds['train'].column_names}")
print(f"  Num rows: {len(ds['train'])}")
print(f"  Feature types:")
for col in ds['train'].column_names:
    sample = ds['train'][0][col]
    if isinstance(sample, (list, np.ndarray)):
        print(f"    {col}: array/list of length {len(sample)}")
    else:
        print(f"    {col}: {type(sample).__name__}")

print(f"\nDataset 2 (ds_collected):")
print(f"  Features: {ds_collected['train'].column_names}")
print(f"  Num rows: {len(ds_collected['train'])}")
print(f"  Feature types:")
for col in ds_collected['train'].column_names:
    sample = ds_collected['train'][0][col]
    if isinstance(sample, (list, np.ndarray)):
        print(f"    {col}: array/list of length {len(sample)}")
    else:
        print(f"    {col}: {type(sample).__name__}")

STRUCTURE COMPARISON

Dataset 1 (ds):
  Features: ['observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
  Num rows: 52970
  Feature types:
    observation.state: array/list of length 8
    action: array/list of length 7
    timestamp: float
    frame_index: int
    episode_index: int
    index: int
    task_index: int

Dataset 2 (ds_collected):
  Features: ['observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
  Num rows: 570
  Feature types:
    observation.state: array/list of length 8
    action: array/list of length 7
    timestamp: float
    frame_index: int
    episode_index: int
    index: int
    task_index: int


In [18]:

# Compare data ranges and statistics
print("\n" + "=" * 80)
print("DATA RANGE & STATISTICS COMPARISON")
print("=" * 80)

for feature in ds['train'].column_names:
    print(f"\n{'─' * 80}")
    print(f"Feature: {feature}")
    print(f"{'─' * 80}")
    
    stats1 = get_feature_stats(ds['train'], feature)
    stats2 = get_feature_stats(ds_collected['train'], feature)
    
    # Create comparison DataFrame
    comparison = pd.DataFrame({
        'Dataset 1 (ds)': [
            stats1.get('type', 'N/A'),
            stats1.get('shape', 'N/A'),
            f"{stats1['min']:.6f}",
            f"{stats1['max']:.6f}",
            f"{stats1['mean']:.6f}",
            f"{stats1['std']:.6f}",
            stats1['num_samples']
        ],
        'Dataset 2 (ds_collected)': [
            stats2.get('type', 'N/A'),
            stats2.get('shape', 'N/A'),
            f"{stats2['min']:.6f}",
            f"{stats2['max']:.6f}",
            f"{stats2['mean']:.6f}",
            f"{stats2['std']:.6f}",
            stats2['num_samples']
        ]
    }, index=['Type', 'Shape', 'Min', 'Max', 'Mean', 'Std', 'Num Samples'])
    
    print(comparison.to_string())
    


DATA RANGE & STATISTICS COMPARISON

────────────────────────────────────────────────────────────────────────────────
Feature: observation.state
────────────────────────────────────────────────────────────────────────────────
                    Dataset 1 (ds) Dataset 2 (ds_collected)
Type                         array                    array
Shape        8 elements per sample    8 elements per sample
Min                      -1.800649                -0.517592
Max                       3.456612                 3.282988
Mean                      0.522650                 0.523942
Std                       1.037779                 1.063556
Num Samples                  52970                      570

────────────────────────────────────────────────────────────────────────────────
Feature: action
────────────────────────────────────────────────────────────────────────────────
                    Dataset 1 (ds) Dataset 2 (ds_collected)
Type                         array                    a